In [1]:
!pip install openmeteo_requests
import openmeteo_requests

In [2]:
class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, step: int = 10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.max_speed:
            raise StopIteration

        self.current_speed = min(self.current_speed + self.step, self.max_speed)
        return self.current_speed


In [3]:
class DecreaseSpeed:
    def __init__(self, current_speed: int, min_speed: int = 0, step: int = 10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.min_speed:
            raise StopIteration

        self.current_speed = max(self.current_speed - self.step, self.min_speed)
        return self.current_speed

In [4]:
class Car:

    cars_on_road = 0

    def __init__(self, max_speed: int, current_speed: int = 0):
        self.max_speed = max_speed
        self.current_speed = current_speed

        self.on_road = current_speed > 0

        if self.on_road:
            Car.cars_on_road += 1


    def accelerate(self, upper_border=None, step=10):

        speed_before = self.current_speed

        if upper_border is not None and 0 < upper_border <= self.max_speed:
            iterator = IncreaseSpeed(self.current_speed, upper_border, step)

            for new_speed in iterator:
                self.current_speed = new_speed
                print(f"INFO: Speed increases by {step}")

        else:
            iterator = IncreaseSpeed(self.current_speed, self.max_speed, step)
            self.current_speed = next(iterator, self.current_speed)

        if not self.on_road and self.current_speed > 0:
            self.on_road = True
            Car.cars_on_road += 1

        print(f"INFO: The speed of this car has been increased from {speed_before} to {self.current_speed}")


    def brake(self, lower_border=None, step=10):

        speed_before = self.current_speed

        if lower_border is not None and lower_border >= 0:
            iterator = DecreaseSpeed(self.current_speed, lower_border, step)

            for new_speed in iterator:
                self.current_speed = new_speed
                print(f"INFO: Speed decreases by {step}")

        else:
            iterator = DecreaseSpeed(self.current_speed, 0, step)
            self.current_speed = next(iterator, self.current_speed)

        print(f"INFO: The speed of this car has been decreased from {speed_before} to {self.current_speed}")


    def parking(self):

        if not self.on_road:
            print("Already parked")
            return

        self.brake(0)

        print("Parking the car...")

        self.on_road = False
        Car.cars_on_road -= 1


    @classmethod
    def total_cars(cls):
        return cls.cars_on_road


    @staticmethod
    def show_weather():

        client = openmeteo_requests.Client()

        params = {
            "latitude": 59.91,
            "longitude": 10.75,
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"]
        }

        response = client.weather_api(
            "https://api.open-meteo.com/v1/forecast",
            params=params
        )[0]

        current = response.Current()

        print(f"Current temperature: {round(current.Variables(0).Value(), 1)} C")
        print(f"Current apparent_temperature: {round(current.Variables(1).Value(), 1)} C")
        print(f"Current rain: {current.Variables(2).Value()} mm")
        print(f"Current wind_speed: {round(current.Variables(3).Value(), 1)} m/s")

In [5]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [6]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [7]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [8]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [9]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [10]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [11]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [12]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current temperature: 9.8 C
Current apparent_temperature: 7.4 C
Current rain: 0.0 mm
Current wind_speed: 5.4 m/s
Total cars on road: 2


In [13]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [14]:
Car.show_weather()

Current temperature: 9.8 C
Current apparent_temperature: 7.4 C
Current rain: 0.0 mm
Current wind_speed: 5.4 m/s
